# Unified Scan Invoice Split

Splits a single scanned PDF containing many invoices into one row per invoice.

A scanned accounts payable batch is rarely one invoice per file. A typical batch is a
cover sheet, then each invoice followed by its supporting documents: signoff sheets,
job work orders, request emails, and job-site photographs. Extracting from the file as
a whole produces one row where there should be several, and extracting from every page
produces rows for documents that are not bills.

This notebook classifies every page first, keeps only the pages that are actually
payable, groups them by invoice number, and derives the amount sign from the document
type so that credit memos land as negative values.

Everything runs in Snowflake with three AI SQL functions and no external services.

**Reference result:** a 19-page batch containing four invoices interleaved with
photographs, emails and handwritten work orders returned exactly four invoice rows,
with a batch total matching the cover sheet to the cent.

## Parameters

Set these three values and the rest of the notebook runs unchanged.

`source_stage` must point at an internal or external stage with a directory table
enabled (`DIRECTORY = (ENABLE = TRUE)`), holding the scanned PDFs. `target_database`
and `target_schema` are where the three output tables get created.

The stage does not have to live in the target schema.

In [ ]:
source_stage    = "@MY_DB.MY_SCHEMA.INVOICE_STAGE"
file_pattern    = "%.pdf"

target_database = "MY_DB"
target_schema   = "MY_SCHEMA"

In [ ]:
%%sql -r set_context
USE SCHEMA {{target_database}}.{{target_schema}}

## 1. One row per page

`AI_PARSE_DOCUMENT` in `LAYOUT` mode returns Markdown that preserves table structure,
which matters because the cover sheet and the invoice totals blocks are both tables.
Setting `page_split` to true returns a `pages` array, so a single `LATERAL FLATTEN`
gives one row per page.

Page indexes returned by the function are zero-based; adding one keeps the page
numbers matching what a human sees in a PDF viewer.

In [ ]:
%%sql -r scan_pages
CREATE OR REPLACE TABLE SCAN_PAGES AS
SELECT
    d.relative_path         AS file_name,
    f.value:index::INT + 1  AS page_num,
    f.value:content::STRING AS page_text
FROM DIRECTORY('{{source_stage}}') d,
     LATERAL FLATTEN(
         input => AI_PARSE_DOCUMENT(
             TO_FILE('{{source_stage}}', d.relative_path),
             {'mode': 'LAYOUT', 'page_split': true}
         ):pages
     ) f
WHERE d.relative_path LIKE '{{file_pattern}}'

In [ ]:
%%sql -r page_preview
SELECT page_num, LENGTH(page_text) AS chars, LEFT(page_text, 120) AS preview
FROM SCAN_PAGES
ORDER BY page_num

## 2. Classify each page

This is the step that does the real work, and it is worth being explicit about why it
cannot be replaced by something simpler.

**Why not filter on text length?** Because it does not separate the classes. In the
sample batch a photograph of a parking lot returned 932 characters while a genuine
email continuation page returned 17. One photo page produced more letters than a real
signoff sheet. The ordering is inverted, so no length threshold works.

**Why not just search for a total?** Because supporting documents carry totals too.
Page 17 of the sample is a handwritten job work order reading `TOTAL AMOUNT $5334.83`,
which is the *subtotal* of the invoice two pages earlier. A pipeline that hunts for
dollar amounts books a phantom fifth invoice here, and the batch total is wrong by
exactly that amount. Every number on that page is real, so nothing downstream catches
it.

The label descriptions below are the control. Only four labels are used, and every one
of them changes what happens downstream: invoice versus credit memo sets the sign,
transmittal identifies the cover sheet carrying the batch's expected totals, and
supporting is excluded. Distinctions that do not change behaviour are not worth
maintaining, so photographs, emails, signoffs and work orders share a single label.

The invoice number is captured twice: once by regex against the printed label and once
by the model during extraction. Comparing them at the end is a free correctness check.

In [ ]:
%%sql -r scan_page_types
CREATE OR REPLACE TABLE SCAN_PAGE_TYPES AS
SELECT
    file_name,
    page_num,
    page_text,
    CASE
        WHEN TRIM(page_text) = '' THEN 'SUPPORTING'
        ELSE AI_CLASSIFY(
            page_text,
            [
                {'label': 'VENDOR_INVOICE',
                 'description': 'A bill from a vendor requesting payment, with its own invoice number, line items and a total amount due.'},
                {'label': 'CREDIT_MEMO',
                 'description': 'A credit or credit note from a vendor that reduces the amount owed, usually labelled credit memo, credit note or credit invoice.'},
                {'label': 'TRANSMITTAL',
                 'description': 'A cover or summary sheet listing several separate invoice numbers and their balances. A batch header, not itself a bill.'},
                {'label': 'SUPPORTING',
                 'description': 'Any page that is not itself a bill: photographs, email correspondence, building services request forms, service signoff sheets, job work orders, and blank or unreadable pages. May show labour, materials, totals or dollar amounts, and may be handwritten, but it documents or accompanies work rather than requesting payment.'}
            ],
            {'task_description': 'Classify one page from a scanned accounts payable batch. The batch contains vendor invoices interleaved with supporting documents. Decide what this page is based only on the page itself.'}
        ):labels[0]::VARCHAR
    END AS page_class,
    REGEXP_SUBSTR(page_text, 'Invoice\\s*(Number|No\\.|No|#)\\s*:?\\s*([0-9]{4,})', 1, 1, 'e', 2) AS invoice_number
FROM SCAN_PAGES

In [ ]:
%%sql -r page_classes
SELECT
    page_class,
    COUNT(*) AS pages,
    LISTAGG(page_num, ', ') WITHIN GROUP (ORDER BY page_num) AS page_list
FROM SCAN_PAGE_TYPES
GROUP BY page_class
ORDER BY page_class

## 3. One row per invoice

Payable pages are grouped by invoice number, so a multi-page invoice collapses into a
single row and its pages are recorded in `source_pages` for traceability back to the
original scan.

`BOOLOR_AGG` means any page identifying as a credit memo types the whole document as
one. The sign is then derived in SQL, never asked of the model:

```sql
CASE WHEN doc_type = 'CREDIT_MEMO' THEN -ABS(total_amount) ELSE ABS(total_amount) END
```

Credit memos are not printed with a minus sign, so the sign cannot be read off the
page. It is a consequence of the document type, and deriving it makes the outcome
deterministic rather than a matter of how the model interpreted the layout.

One note on the `vendor_name` prompt. Invoices that have been factored carry a
remittance stamp naming the finance company, often printed directly over the totals
block. Without an explicit instruction to ignore it, extraction returns the factor
instead of the vendor, and the payee is wrong. The guard clause is in the field
description below.

Rows land as `needs_review` so nothing reaches the ERP without a human release.

In [ ]:
%%sql -r scan_invoices
CREATE OR REPLACE TABLE SCAN_INVOICES AS
WITH grouped_pages AS (
    SELECT
        file_name,
        invoice_number,
        CASE WHEN BOOLOR_AGG(page_class = 'CREDIT_MEMO')
             THEN 'CREDIT_MEMO' ELSE 'VENDOR_INVOICE' END        AS doc_type,
        ARRAY_AGG(page_num) WITHIN GROUP (ORDER BY page_num)      AS source_pages,
        LISTAGG(page_text, '\n') WITHIN GROUP (ORDER BY page_num) AS invoice_text
    FROM SCAN_PAGE_TYPES
    WHERE page_class IN ('VENDOR_INVOICE', 'CREDIT_MEMO')
      AND invoice_number IS NOT NULL
    GROUP BY file_name, invoice_number
),
extracted AS (
    SELECT
        file_name,
        invoice_number,
        doc_type,
        source_pages,
        AI_EXTRACT(
            text => invoice_text,
            responseFormat => {
                'vendor_name': 'The company issuing this invoice. Not the customer being billed, and not any factoring or finance company named in a remittance notice.',
                'invoice_number': 'The invoice number printed on this document.',
                'invoice_date': 'The invoice date in YYYY-MM-DD format. Not the due date.',
                'customer_po': 'The customer PO number printed on this invoice.',
                'total_amount': 'The final total amount due, as a plain number with no currency symbol or thousands separator.'
            }
        ):response AS r
    FROM grouped_pages
),
typed AS (
    SELECT
        file_name,
        invoice_number,
        doc_type,
        source_pages,
        r:vendor_name::STRING                                                        AS vendor_name,
        r:invoice_number::STRING                                                     AS extracted_invoice_number,
        TRY_TO_DATE(r:invoice_date::STRING)                                          AS invoice_date,
        r:customer_po::STRING                                                        AS customer_po,
        TRY_TO_NUMBER(REGEXP_REPLACE(r:total_amount::STRING, '[^0-9.-]', ''), 12, 2) AS total_amount
    FROM extracted
)
SELECT
    file_name,
    invoice_number,
    doc_type,
    vendor_name,
    extracted_invoice_number,
    invoice_date,
    customer_po,
    total_amount,
    CASE WHEN doc_type = 'CREDIT_MEMO'
         THEN -ABS(total_amount) ELSE ABS(total_amount) END AS signed_amount,
    source_pages,
    'needs_review'                                          AS status
FROM typed

In [ ]:
%%sql -r invoices
SELECT invoice_number, doc_type, vendor_name, invoice_date,
       customer_po, signed_amount, source_pages, status
FROM SCAN_INVOICES
ORDER BY file_name, invoice_number

## 4. Controls

Three numbers worth watching on every batch.

`payable_pages_without_number` counts pages classified as payable where no invoice
number could be read. These are silently dropped from the output, so a non-zero value
means an invoice is missing.

`invoice_number_disagreements` compares the regex reading against the model's. They
are independent signals, so disagreement points at a page worth looking at.

`batch_total` is the strongest check available and needs no knowledge of the pipeline:
it should equal the total on the cover sheet.

In [ ]:
%%sql -r controls
SELECT
    (SELECT COUNT(*) FROM SCAN_PAGES)                      AS pages_parsed,
    (SELECT COUNT(*) FROM SCAN_INVOICES)                   AS invoices_found,
    (SELECT SUM(signed_amount) FROM SCAN_INVOICES)         AS batch_total,
    (SELECT COUNT(*) FROM SCAN_PAGE_TYPES
      WHERE page_class IN ('VENDOR_INVOICE', 'CREDIT_MEMO')
        AND invoice_number IS NULL)                        AS payable_pages_without_number,
    (SELECT COUNT(*) FROM SCAN_INVOICES
      WHERE invoice_number <> extracted_invoice_number)    AS invoice_number_disagreements

## 5. Credit memo sign

Credit memos are not printed with a minus sign, so the sign cannot be read off the
page — it is a consequence of the document type. That makes it worth testing directly.

If the batch you ran contains no credit memo, the query below forces the type onto the
lowest-numbered invoice to demonstrate the arithmetic. It reads from `SCAN_INVOICES`
without modifying it.

Expect the forced row to flip negative and the batch total to fall by twice that row's
value. To test against real data, drop a credit memo into the stage and re-run the
notebook from the top; the classifier types it and the sign follows automatically.

In [ ]:
%%sql -r credit_memo_sign
WITH forced AS (
    SELECT
        invoice_number,
        CASE WHEN invoice_number = (SELECT MIN(invoice_number) FROM SCAN_INVOICES)
             THEN 'CREDIT_MEMO' ELSE doc_type END AS doc_type,
        total_amount,
        signed_amount AS original_signed_amount
    FROM SCAN_INVOICES
)
SELECT
    invoice_number,
    doc_type,
    original_signed_amount,
    CASE WHEN doc_type = 'CREDIT_MEMO'
         THEN -ABS(total_amount) ELSE ABS(total_amount) END AS resigned_amount,
    SUM(CASE WHEN doc_type = 'CREDIT_MEMO'
             THEN -ABS(total_amount) ELSE ABS(total_amount) END) OVER () AS batch_total_with_credit,
    SUM(original_signed_amount) OVER ()                                 AS batch_total_all_invoices
FROM forced
ORDER BY invoice_number